# pvsamlab BESS Sandbox

Demonstrates all four simulation modes after the Phase 3 BESS extension.

| Cell | Mode | PySAM module |
|------|------|--------------|
| 1 | Setup & imports | — |
| 2 | PV-only (`System`) | `Pvsamv1 en_batt=0` |
| 3 | PV + BESS (`PvBessSystem`) | `Pvsamv1 en_batt=1` |
| 4 | BESS-only (`StandaloneBessSystem`) | `StandAloneBattery` |
| 5 | Financial overlay (`compute_lcoe` + `compute_lcos`) | `SingleOwner` / `Cashloan` |

> **Weather data:** Uses cached NSRDB files for Scurry County, TX  
> `lat=33.0278, lon=-100.0814, year='2017'`  
> so no API calls are required to run this notebook.

In [1]:
import json
import pprint

from pvsamlab import (
    # PV-only
    System,
    TrackingMode,
    # BESS
    Battery,
    BessDispatch,
    PvBessSystem,
    StandaloneBessSystem,
    # Financial
    Financial,
    compute_lcoe,
    compute_lcos,
    compute_npv,
    compute_irr,
)

# ---------------------------------------------------------------------------
# Shared site parameters — reused in every cell below
# ---------------------------------------------------------------------------
SITE_LAT   = 33.0278
SITE_LON   = -100.0814
MET_YEAR   = '2017'           # cached NSRDB file — no download needed
TARGET_MW  = 100_000          # 100 MWac target plant size

print('pvsamlab imported successfully.')

pvsamlab imported successfully.


---
## Cell 2 — PV-only (`System`)

Verifies that the existing PV-only path is unchanged after the BESS extension.

In [2]:
# Build the PV-only system
pv_plant = System(
    lat=SITE_LAT,
    lon=SITE_LON,
    target_kwac=TARGET_MW,
    target_dcac=1.30,
    met_year=MET_YEAR,
    tracking_mode=TrackingMode.SAT,
)

print(f'System built:  {pv_plant.kwac:,.0f} kWac  |  DC/AC = {pv_plant.dc_ac_ratio:.3f}')

# Run simulation
pv_results = pv_plant.run()

print('\n--- PV-only results ---')
for k, v in pv_results.items():
    print(f'  {k:<50s} {v}')

System built:  100,000 kWac  |  DC/AC = 1.300

--- PV-only results ---
  met_year                                           nsrdb_33.0278_-100.0814_nsrdb-GOES-aggregated-v4-0-0_60_2017.csv
  voc_max                                            1396.99
  performance_ratio                                  87.7
  annual_ghi                                         1931562.0
  annual_poa_front                                   1395862784.606
  annual_poa_eff                                     1449256358.674
  Nominal POA Irradiance                             1464795691.533
  annual_poa_beam_nom                                1102286459.581
  annual_poa_shading_loss_percent                    1.47305
  annual_poa_soiling_loss_percent                    2.5
  annual_poa_iam_loss_percent                        0.801289
  annual_rear_ground_reflected_percent               4.16051
  annual_bifacial_electrical_mismatch_percent        0.16879
  annual_dc_nominal                                  33

---
## Cell 3 — PV + BESS (`PvBessSystem`)

Adds a 4-hour LFP battery co-located with the PV plant.  
Uses `Pvsamv1` with `en_batt=1` and **self-consumption** dispatch.

In [3]:
# Battery: 400 MWh / 100 MW (4-hour duration), LFP, DC-coupled
batt = Battery(
    chemistry='LFP',
    energy_kwh=400_000,       # 400 MWh
    power_kw=100_000,         # 100 MW
    soc_min=10.0,
    soc_max=95.0,
    coupling='DC',
)

# Dispatch: automated self-consumption (charges from excess PV, discharges to load)
disp = BessDispatch(strategy='self_consumption')

# Load profile: flat 80 MW demand — gives self-consumption a discharge signal.
# Without a load profile the battery has no demand to discharge against.
load_profile_kw = [80_000.0] * 8760

# Build PV+BESS system — inherits all PV fields from System
pvbess_plant = PvBessSystem(
    lat=SITE_LAT,
    lon=SITE_LON,
    target_kwac=TARGET_MW,
    target_dcac=1.30,
    met_year=MET_YEAR,
    tracking_mode=TrackingMode.SAT,
    battery=batt,
    dispatch=disp,
    load_profile=load_profile_kw,
)

print(f'PvBessSystem built:  {pvbess_plant.kwac:,.0f} kWac  |  '
      f'Battery: {batt.energy_kwh:,.0f} kWh / {batt.power_kw:,.0f} kW')

# Run simulation
pvbess_results = pvbess_plant.run()

print('\n--- PV+BESS results ---')
for k, v in pvbess_results.items():
    print(f'  {k:<50s} {v}')


PvBessSystem built:  100,000 kWac  |  Battery: 400,000 kWh / 100,000 kW

--- PV+BESS results ---
  met_year                                           nsrdb_33.0278_-100.0814_nsrdb-GOES-aggregated-v4-0-0_60_2017.csv
  voc_max                                            1396.99
  performance_ratio                                  88.4
  annual_ghi                                         1931562.0
  annual_poa_front                                   1395862784.606
  annual_poa_eff                                     1449256358.674
  Nominal POA Irradiance                             1464795691.533
  annual_poa_beam_nom                                1102286459.581
  annual_poa_shading_loss_percent                    1.47305
  annual_poa_soiling_loss_percent                    2.5
  annual_poa_iam_loss_percent                        0.801289
  annual_rear_ground_reflected_percent               4.16051
  annual_bifacial_electrical_mismatch_percent        0.16879
  annual_dc_nominal          

---
## Cell 4 — BESS-only (`StandaloneBessSystem`)

No PV.  Simulates a standalone battery against a flat 8760-hour load profile.  
Uses `PySAM.StandAloneBattery`.  Downloads ambient temperature from the same
cached NSRDB file.

In [4]:
import numpy as np

# Flat load profile: 50 MW constant for all 8760 hours
flat_load_kw = [50_000.0] * 8760

# Battery: 200 MWh / 50 MW (4-hour), LFP, AC-coupled (standalone requires AC)
sa_batt = Battery(
    chemistry='LFP',
    energy_kwh=200_000,
    power_kw=50_000,
    soc_min=10.0,
    soc_max=95.0,
)

# Dispatch: self-consumption — the Battery standalone module's manual dispatch
# (batt_dispatch_choice=0) is non-functional in PySAM 6 standalone mode;
# self-consumption (choice=3) correctly cycles the battery against the load profile.
sa_disp = BessDispatch(strategy='self_consumption')

standalone = StandaloneBessSystem(
    battery=sa_batt,
    dispatch=sa_disp,
    load_profile=flat_load_kw,
    lat=SITE_LAT,
    lon=SITE_LON,
    weather_year=MET_YEAR,
)

print(f'StandaloneBessSystem built:  '
      f'{sa_batt.energy_kwh:,.0f} kWh / {sa_batt.power_kw:,.0f} kW')

# Run simulation (downloads/uses cached ambient temperature)
sa_results = standalone.run()

print('\n--- BESS-only results ---')
for k, v in sa_results.items():
    print(f'  {k:<50s} {v}')


StandaloneBessSystem built:  200,000 kWh / 50,000 kW

--- BESS-only results ---
  batt_annual_discharge_energy_kwh                   87529144.912
  batt_annual_charge_energy_kwh                      95739125.474
  batt_roundtrip_efficiency_pct                      91.47
  batt_capacity_end_of_life_pct                      94.1
  batt_capacity_percent                              [100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 99.99128091679363, 99.99128091679363, 99.99128091679363, 99.99128091679363, 99.99128091679363, 99.99128091679363, 99.99128091679363, 99.99128091679363, 99.99128091679363, 99.99128091679363, 99.99128091679363, 99.99128091679363, 99.99128091679363, 99.99128091679363, 99.99128091679363, 99.99128091679363, 99.99128091679363, 99.99128091679363, 99.99128091679363, 99.99128091679363, 99.99128091679363, 99.99128091679363, 99.99128091679363, 99.99128091679363, 99.99128091679

---
## Cell 5 — Financial overlay

Demonstrates `compute_lcoe` (PySAM SingleOwner / Cashloan) and `compute_lcos`
(pure-Python post-simulation) on top of the already-executed simulations.

For the LCOS calculation we construct a simple annual discharge projection:
year-1 discharge from the PV+BESS simulation, degraded by the battery
calendar degradation rate for each subsequent year.

In [5]:
# ── Financial assumptions ──────────────────────────────────────────────────
fin = Financial(
    analysis_period=25,
    pv_capex_per_kwdc=700.0,      # $/kWdc
    opex_per_kwac_year=15.0,      # $/kWac/yr
    ppa_rate=30.0,                # $/MWh — gives PV-only IRR ~12-13 % (in target range)
    ppa_escalation=1.0,           # %/yr
    discount_rate=8.0,            # % real
    inflation_rate=2.5,
    debt_fraction=0.0,        # all-equity: avoids ITC-covers-all-equity artefact
    loan_rate=5.0,
    loan_term=18,
    federal_tax_rate=21.0,
    itc_rate=0.0,   # no ITC simplifies equity analysis; IRR driven by PPA/capex only
    depreciation_schedule='MACRS5',
)

# ── LCOE / IRR / NPV  (PySAM SingleOwner — kwac > 1000 kW) ────────────────
print('Running compute_lcoe on PV-only system...')
lcoe_pv = compute_lcoe(pv_plant, fin)

print('\n--- PV-only financials (SingleOwner) ---')
for k, v in lcoe_pv.items():
    print(f'  {k:<40s} {v}')

# ── LCOE on PV+BESS system ─────────────────────────────────────────────────
print('\nRunning compute_lcoe on PV+BESS system...')
lcoe_pvbess = compute_lcoe(pvbess_plant, fin)

print('\n--- PV+BESS financials (SingleOwner) ---')
for k, v in lcoe_pvbess.items():
    print(f'  {k:<40s} {v}')

# ── LCOS (post-simulation Python) ─────────────────────────────────────────
# Project annual discharge for 25 years using calendar degradation
y1_discharge_kwh = pvbess_results['batt_annual_discharge_energy_kwh']
deg_rate = batt.calendar_degradation / 100.0   # fraction per year
annual_discharge = [
    y1_discharge_kwh * (1.0 - deg_rate) ** yr
    for yr in range(fin.analysis_period)
]

annual_bess_opex = batt.energy_kwh * batt.opex_per_kwh_year   # $/year
dr = fin.discount_rate / 100.0

lcos = compute_lcos(
    battery=batt,
    annual_discharge_kwh=annual_discharge,
    annual_opex=annual_bess_opex,
    replacement_events=[],   # no replacement assumed
    discount_rate=dr,
)

print(f'\n--- LCOS ---')
print(f'  Year-1 discharge           {y1_discharge_kwh:>15,.1f}  kWh')
print(f'  Annual BESS O&M            {annual_bess_opex:>15,.0f}  $/yr')
print(f'  LCOS                       {lcos:>15.4f}  $/kWh')

# ── Standalone NPV / IRR helpers ──────────────────────────────────────────
# Quick sanity-check using the pure-Python helpers with a toy cash-flow series
example_cashflows = [-1_000_000] + [120_000] * 25    # $1M capex, $120k/yr
ex_npv = compute_npv(example_cashflows, discount_rate=0.08)
ex_irr = compute_irr(example_cashflows)

print(f'\n--- Standalone NPV / IRR (toy example) ---')
print(f'  Cash flows: -$1M upfront, +$120k/yr × 25 years')
print(f'  NPV @ 8%   {ex_npv:>12,.2f}  $')
print(f'  IRR        {ex_irr * 100:>12.3f}  %')

Running compute_lcoe on PV-only system...

--- PV-only financials (SingleOwner) ---
  lcoe_real_cents_per_kwh                  2.2765
  lcoe_nom_cents_per_kwh                   2.8084
  npv_usd                                  10182382.2
  npv_sam_usd                              10182382.2
  annual_revenue_bonus_usd                 0.0
  irr_pct                                  12.821
  total_installed_cost_usd                 90999783.0

Running compute_lcoe on PV+BESS system...

--- PV+BESS financials (SingleOwner) ---
  lcoe_real_cents_per_kwh                  5.4519
  lcoe_nom_cents_per_kwh                   6.7256
  npv_usd                                  -86642945.34
  npv_sam_usd                              -86642945.34
  annual_revenue_bonus_usd                 0.0
  irr_pct                                  1.723
  total_installed_cost_usd                 205999783.0

--- LCOS ---
  Year-1 discharge              43,937,277.7  kWh
  Annual BESS O&M                  3,200,000 